# Chapter 12 — One Operation, Several Model APIs

**Companion to Applied AI**

Question: Can one operation survive three genuinely different provider dialects?

By the end of this notebook you will have:

- created three fake provider response shapes
- normalized them behind a single operation
- shown provider payload differences do not leak downstream

## What this notebook demonstrates
Three invented-but-plausible provider dialects behind one `classify` operation. No API keys; no real providers.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. Three dialects, same intent

In [2]:
def luna_responses(prompt: str) -> dict:   # nested output_text + usage
    return {"kind": "responses", "output_text": "LABEL: needs_fix",
            "usage": {"in": 38, "out": 12}}

def mimo_chat(prompt: str) -> dict:          # choices/message list
    return {"kind": "chat", "choices": [{"message": {"content": "needs_fix"}}],
            "usage": {"prompt_tokens": 279, "completion_tokens": 3}}

def minimax_messages(prompt: str) -> dict:   # content blocks
    return {"kind": "messages", "content": [{"type": "text", "text": "{\"label\": \"needs_fix\"}"}],
            "usage": {"input": 73, "output": 9}}

for f in (luna_responses, mimo_chat, minimax_messages):
    print(f.__name__, "->", f("review row 14"))

luna_responses -> {'kind': 'responses', 'output_text': 'LABEL: needs_fix', 'usage': {'in': 38, 'out': 12}}
mimo_chat -> {'kind': 'chat', 'choices': [{'message': {'content': 'needs_fix'}}], 'usage': {'prompt_tokens': 279, 'completion_tokens': 3}}
minimax_messages -> {'kind': 'messages', 'content': [{'type': 'text', 'text': '{"label": "needs_fix"}'}], 'usage': {'input': 73, 'output': 9}}


## 2. One operation: adapters at the edge, canonical inside

In [3]:
import json
def adapt(raw: dict) -> str:
    if raw["kind"] == "responses":
        return raw["output_text"].split("LABEL:")[1].strip()
    if raw["kind"] == "chat":
        return raw["choices"][0]["message"]["content"].strip()
    if raw["kind"] == "messages":
        return json.loads(raw["content"][0]["text"])["label"]
    raise ValueError("unknown dialect")

def classify(prompt: str, provider) -> dict:
    return {"label": adapt(provider(prompt)), "prompt": prompt}

outs = [classify("review row 14", p) for p in (luna_responses, mimo_chat, minimax_messages)]
print(outs)
assert {o["label"] for o in outs} == {"needs_fix"}

[{'label': 'needs_fix', 'prompt': 'review row 14'}, {'label': 'needs_fix', 'prompt': 'review row 14'}, {'label': 'needs_fix', 'prompt': 'review row 14'}]


## Interpretation
- Supports: dialect handling belongs in thin adapters; downstream code sees one canonical shape.
- Does NOT support: claims about real vendors' APIs.

## Try it yourself
1. Add a fourth dialect that nests the label one level deeper.
2. Make one provider return an apology instead of a label and watch `adapt` raise — then decide the operation's refusal shape.
3. Move usage extraction into the adapters (see chapter 13).